In [ ]:
# Mount into drive

from google.colab import drive

drive.mount("/content/drive")

%cd '/content/drive/MyDrive/assignment3_NLP_spr26/'

!pip install -r requirements.txt




In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Configuration
NUM_SAMPLES = 1000  # Sample dataset size
TEXT_EMBED_DIM = 768  # Typical BERT embedding size
IMAGE_EMBED_DIM = 2048  # Typical ResNet embedding size
RATING_RANGE = (0, 5)  # Movie rating scale 0-5
BATCH_SIZE = 32
LEARNING_RATE = 0.001

print("Configuration set up for movie rating prediction!")

In [ ]:
# Generate random embeddings for movie rating prediction
def generate_random_movie_data(num_samples, text_dim, image_dim, rating_range):
    """Generate random text, image embeddings and user ratings"""
    
    # Random text embeddings (simulating BERT/CLIP text embeddings from movie descriptions)
    text_embeddings = torch.randn(num_samples, text_dim)
    
    # Random image embeddings (simulating ResNet/ViT embeddings from movie posters)
    image_embeddings = torch.randn(num_samples, image_dim)
    
    # Generate realistic user ratings (0-5 scale)
    # Create a more realistic distribution with some bias towards middle ratings
    ratings = torch.zeros(num_samples)
    
    for i in range(num_samples):
        # Simulate realistic rating distribution (more 3-4 star ratings)
        rating = np.random.normal(3.5, 1.2)  # Center around 3.5 with some variance
        rating = np.clip(rating, rating_range[0], rating_range[1])
        ratings[i] = rating
    
    # Add some correlation between embeddings and ratings for more realistic simulation
    # Higher quality embeddings (by magnitude) tend to have higher ratings
    text_quality = torch.norm(text_embeddings, dim=1)
    image_quality = torch.norm(image_embeddings, dim=1)
    combined_quality = (text_quality + image_quality) / 2
    
    # Adjust ratings based on embedding quality
    quality_adjustment = (combined_quality - combined_quality.mean()) * 0.3
    ratings = torch.clamp(ratings + quality_adjustment, rating_range[0], rating_range[1])
    
    return text_embeddings, image_embeddings, ratings

# Generate sample data
text_embeds, image_embeds, user_ratings = generate_random_movie_data(
    NUM_SAMPLES, TEXT_EMBED_DIM, IMAGE_EMBED_DIM, RATING_RANGE
)

print(f"Generated {NUM_SAMPLES} movie samples:")
print(f"Text embeddings shape: {text_embeds.shape}")
print(f"Image embeddings shape: {image_embeds.shape}")
print(f"User ratings shape: {user_ratings.shape}")
print(f"Rating statistics:")
print(f"  Mean: {user_ratings.mean():.2f}")
print(f"  Std: {user_ratings.std():.2f}")
print(f"  Min: {user_ratings.min():.2f}")
print(f"  Max: {user_ratings.max():.2f}")
print(f"  Rating distribution: {torch.bincount(user_ratings.long())}")

In [ ]:
class MovieRatingDataset(Dataset):
    """Dataset class for multi-modal movie rating prediction"""
    
    def __init__(self, text_embeddings, image_embeddings, user_ratings):
        self.text_embeddings = text_embeddings
        self.image_embeddings = image_embeddings
        self.user_ratings = user_ratings
        
    def __len__(self):
        return len(self.text_embeddings)
    
    def __getitem__(self, idx):
        return {
            'text': self.text_embeddings[idx],
            'image': self.image_embeddings[idx],
            'rating': self.user_ratings[idx]
        }

def early_fusion(text_embeds, image_embeds):
    """
    Early Fusion: Concatenate text and image embeddings
    
    Args:
        text_embeds: Tensor of shape (batch_size, text_dim)
        image_embeds: Tensor of shape (batch_size, image_dim)
    
    Returns:
        fused_embeds: Tensor of shape (batch_size, text_dim + image_dim)
    """
    # Normalize embeddings to prevent scale imbalance
    text_embeds = nn.functional.normalize(text_embeds, p=2, dim=1)
    image_embeds = nn.functional.normalize(image_embeds, p=2, dim=1)
    
    # Concatenate along feature dimension
    fused_embeds = torch.cat([text_embeds, image_embeds], dim=1)
    
    return fused_embeds

# Test early fusion
sample_text = text_embeds[:4]
sample_image = image_embeds[:4]
fused = early_fusion(sample_text, sample_image)

print(f"Early Fusion Test:")
print(f"Text shape: {sample_text.shape}")
print(f"Image shape: {sample_image.shape}")
print(f"Fused shape: {fused.shape}")
print(f"Fused dimension: {TEXT_EMBED_DIM + IMAGE_EMBED_DIM}")

In [ ]:
class EarlyFusionRatingPredictor(nn.Module):
    """Neural network for movie rating prediction using early fusion"""
    
    def __init__(self, text_dim, image_dim, hidden_dims=[1024, 512, 256, 128]):
        super(EarlyFusionRatingPredictor, self).__init__()
        
        # Input dimension after fusion
        fused_dim = text_dim + image_dim
        
        # Build the regression layers
        layers = []
        prev_dim = fused_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.BatchNorm1d(hidden_dim)
            ])
            prev_dim = hidden_dim
        
        # Output layer for regression (single value for rating prediction)
        layers.append(nn.Linear(prev_dim, 1))
        
        # Add activation to constrain output to rating range
        self.rating_min = 0.0
        self.rating_max = 5.0
        
        self.regressor = nn.Sequential(*layers)
        
    def forward(self, text_embeds, image_embeds):
        # Apply early fusion
        fused_embeds = early_fusion(text_embeds, image_embeds)
        
        # Pass through regressor
        rating_logits = self.regressor(fused_embeds)
        
        # Apply sigmoid and scale to rating range [0, 5]
        rating_pred = torch.sigmoid(rating_logits) * (self.rating_max - self.rating_min) + self.rating_min
        
        return rating_pred.squeeze()

# Initialize the model
model = EarlyFusionRatingPredictor(
    text_dim=TEXT_EMBED_DIM,
    image_dim=IMAGE_EMBED_DIM
)

print(f"Rating prediction model initialized with {sum(p.numel() for p in model.parameters())} parameters")
print(model)

In [ ]:
# Split data into train and test sets
train_size = int(0.8 * NUM_SAMPLES)
test_size = NUM_SAMPLES - train_size

train_dataset = MovieRatingDataset(
    text_embeds[:train_size],
    image_embeds[:train_size],
    user_ratings[:train_size]
)

test_dataset = MovieRatingDataset(
    text_embeds[train_size:],
    image_embeds[train_size:],
    user_ratings[train_size:]
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training set size: {len(train_dataset)}")
print(f"Test set size: {len(test_dataset)}")
print(f"Number of training batches: {len(train_loader)}")
print(f"Training rating stats: Mean={train_dataset.user_ratings.mean():.2f}, Std={train_dataset.user_ratings.std():.2f}")
print(f"Test rating stats: Mean={test_dataset.user_ratings.mean():.2f}, Std={test_dataset.user_ratings.std():.2f}")

In [ ]:
# Training setup for regression
criterion = nn.MSELoss()  # Mean Squared Error for regression
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
num_epochs = 15

# Training loop for rating prediction
def train_rating_model(model, train_loader, criterion, optimizer, num_epochs):
    model.train()
    train_losses = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_batches = 0
        
        for batch in train_loader:
            text = batch['text']
            image = batch['image']
            ratings = batch['rating']
            
            # Forward pass
            predictions = model(text, image)
            loss = criterion(predictions, ratings)
            
            # Backward pass and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            num_batches += 1
        
        avg_loss = epoch_loss / num_batches
        train_losses.append(avg_loss)
        
        print(f'Epoch [{epoch+1}/{num_epochs}], MSE Loss: {avg_loss:.4f}')
    
    return train_losses

# Train the model
print("Starting rating prediction training...")
train_losses = train_rating_model(model, train_loader, criterion, optimizer, num_epochs)
print("Training completed!")

In [ ]:
# Evaluation for rating prediction
def evaluate_rating_model(model, test_loader):
    model.eval()
    all_predictions = []
    all_actuals = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in test_loader:
            text = batch['text']
            image = batch['image']
            ratings = batch['rating']
            
            predictions = model(text, image)
            loss = criterion(predictions, ratings)
            total_loss += loss.item()
            
            all_predictions.extend(predictions.cpu().numpy())
            all_actuals.extend(ratings.cpu().numpy())
    
    # Calculate metrics
    all_predictions = np.array(all_predictions)
    all_actuals = np.array(all_actuals)
    
    mse = mean_squared_error(all_actuals, all_predictions)
    mae = mean_absolute_error(all_actuals, all_predictions)
    rmse = np.sqrt(mse)
    
    # Calculate R-squared
    ss_res = np.sum((all_actuals - all_predictions) ** 2)
    ss_tot = np.sum((all_actuals - np.mean(all_actuals)) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    
    avg_loss = total_loss / len(test_loader)
    
    return avg_loss, mse, mae, rmse, r2, all_predictions, all_actuals

# Evaluate the model
test_loss, mse, mae, rmse, r2, predictions, actuals = evaluate_rating_model(model, test_loader)

print(f'Rating Prediction Results:')
print(f'Test MSE Loss: {test_loss:.4f}')
print(f'Mean Squared Error: {mse:.4f}')
print(f'Mean Absolute Error: {mae:.4f}')
print(f'Root Mean Squared Error: {rmse:.4f}')
print(f'R-squared: {r2:.4f}')

# Show sample predictions
print(f'\nSample Predictions (Actual vs Predicted):')
sample_indices = np.random.choice(len(predictions), 10, replace=False)
for i in sample_indices:
    actual = actuals[i]
    predicted = predictions[i]
    error = abs(actual - predicted)
    print(f'  Movie {i+1}: {actual:.2f}★ → {predicted:.2f}★ (Error: {error:.2f})')

# Rating distribution analysis
print(f'\nRating Distribution Analysis:')
print(f'Actual ratings - Mean: {np.mean(actuals):.2f}, Std: {np.std(actuals):.2f}')
print(f'Predicted ratings - Mean: {np.mean(predictions):.2f}, Std: {np.std(predictions):.2f}')